# Análise de Projetos de Investimento - Distrito Federal

## 1. Introdução e Contexto

### Documentação de Configurações

### Fonte dos Dados

Os dados utilizados nesta análise foram extraídos da **API ObrasGov.br**, especificamente do endpoint `/projeto-investimento`.

- **URL Base da API:** `https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento`
- **Filtro Aplicado:** A extração foi restrita aos projetos localizados na **Unidade Federativa (UF) do Distrito Federal (DF)**, utilizando o parâmetro `uf=DF`.
- **Formato:** Os dados são retornados em formato JSON, com estrutura aninhada (listas de dicionários) para campos como executores, fontes de recurso e tipos de projeto.

### Objetivo da Análise

O objetivo principal desta análise é demonstrar a capacidade de construir um pipeline de processamento de dados (ETL - Extração, Transformação e Carga) e realizar uma análise exploratória sobre dados públicos de projetos de investimento.

**Objetivos Específicos:**

1. Integrar-se com uma API pública (ObrasGov.br) para extrair dados de forma paginada.
2. Tratar e normalizar dados complexos.
3. Persistir os dados tratados em um banco de dados relacional (PostgreSQL).
4. Gerar insights e visualizações que respondam a perguntas de negócio sobre os investimentos no DF.




## 2. Extração de dados
### 2.1 Importações 


In [8]:
# Importar bibliotecas necessárias
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import time
from datetime import datetime
import json
from tqdm import tqdm


### 2.2 Definição de constantes e funções

In [6]:
# 1. Definir constantes (URL base, parâmetros)
BASE_URL = "https://api.obrasgov.gestao.gov.br/obrasgov/api/projeto-investimento"
DEFAULT_PARAMS = {
    "uf": "DF",
    "size": 1000, # Tamanho máximo por página
    "page": 0
}
MAX_RETRIES = 3
RETRY_DELAY = 5 # segundos

# 2. Função para fazer requisição com retry
def fetch_page_with_retry(page_num, max_retries=MAX_RETRIES, retry_delay=RETRY_DELAY):
    '''Faz a requisição para uma página específica da API com lógica de retry.'''
    params = DEFAULT_PARAMS.copy()
    params['page'] = page_num
    
    for attempt in range(max_retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status() # Levanta exceção para códigos de erro HTTP
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Erro na requisição da página {page_num} (Tentativa {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                print(f"Falha final ao buscar a página {page_num} após {max_retries} tentativas.")
                return None

# 3. Função para paginação
def fetch_all_data():
    '''Implementa a lógica de paginação para extrair todos os dados.'''
    all_data = []
    page_num = 0
    total_pages = 1 # Inicializa com 1 para entrar no loop

    print("\nIniciando extração de dados da API ObrasGov.br (UF=DF)...")

    # Tenta buscar a primeira página para obter o total de páginas
    initial_data = fetch_page_with_retry(0)
    if initial_data and 'content' in initial_data:
        all_data.extend(initial_data['content'])
        total_pages = initial_data.get('totalPages', 1)
        page_num = 1
    elif initial_data is None:
        print("Não foi possível obter a primeira página. Abortando extração.")
        return []
    else:
        print("A primeira página não contém dados ou tem formato inesperado. Abortando extração.")
        return []

    # Continua a busca para as páginas restantes
    with tqdm(total=total_pages, initial=1, desc="Páginas", unit="pág") as pbar:
        while page_num < total_pages:
            data = fetch_page_with_retry(page_num)
            
            if data and 'content' in data:
                all_data.extend(data['content'])
                page_num += 1
                pbar.update(1)
            else:
                print(f"Parando a extração na página {page_num} devido a erro ou falta de dados.")
                break
                
    print(f"Extração concluída. Total de {len(all_data)} registros obtidos.")
    return all_data

### 2.3 Extração e salvamento dos dados

In [9]:
# Cria o diretório para salvar os dados brutos, se não existir
os.makedirs("data/raw", exist_ok=True)

# 4. Executar extração completa
dados_brutos = fetch_all_data()

# Salvar dados brutos
if dados_brutos:
    raw_file_path = 'data/raw/projetos_df_raw.json'
    with open(raw_file_path, 'w', encoding='utf-8') as f:
        json.dump(dados_brutos, f, ensure_ascii=False, indent=4)
    print(f"\nDados brutos salvos em: {raw_file_path}")
    print(f"Total de registros: {len(dados_brutos)}")
    
    # Mostrar amostra dos dados
    print("\nAmostra do primeiro registro:")
    # Usamos [0] para pegar o primeiro elemento da lista de projetos
    print(json.dumps(dados_brutos[0], indent=2)) 
else:
    print("Nenhum dado para salvar.")


Iniciando extração de dados da API ObrasGov.br (UF=DF)...


Páginas: 100%|██████████| 1/1 [00:00<?, ?pág/s]

Extração concluída. Total de 10 registros obtidos.

Dados brutos salvos em: data/raw/projetos_df_raw.json
Total de registros: 10

Amostra do primeiro registro:
{
  "idUnico": "50379.53-54",
  "nome": "DL - 304/2024 - Contrata\u00e7\u00e3o de institui\u00e7\u00e3o para execu\u00e7\u00e3o de servi\u00e7os t\u00e9cnico-especializados para realiza\u00e7\u00e3o de atualiza\u00e7\u00f5es no M\u00e9todo de Dimensionamento de Pavimentos R\u00edgidos do DNI",
  "cep": null,
  "endereco": null,
  "descricao": "Contrata\u00e7\u00e3o de institui\u00e7\u00e3o para execu\u00e7\u00e3o de servi\u00e7os t\u00e9cnico-especializados para realiza\u00e7\u00e3o de atualiza\u00e7\u00f5es no M\u00e9todo de Dimensionamento de Pavimentos R\u00edgidos do DNI",
  "funcaoSocial": "Amplia\u00e7\u00e3o da capacidade de trafego visando a melhoria da seguran\u00e7a do usu\u00e1rio",
  "metaGlobal": "Projetos B\u00e1sicos e Executivos de Engenharia",
  "dataInicialPrevista": "2024-12-20",
  "dataFinalPrevista": "2027-1

# 3. Análise Exploratória
### 3.1 - Carregar Dados em DataFrame
Objetivo: Converter a lista de dicionários extraída da API (dados_brutos) em um objeto DataFrame do Pandas para manipulação e análise.




In [10]:
# Tenta carregar a variável 'dados_brutos' da memória, ou recarrega do arquivo JSON
try:
    # Verifica se a variável 'dados_brutos' existe e não está vazia
    if 'dados_brutos' not in locals() or not dados_brutos:
        with open('data/raw/projetos_df_raw.json', 'r', encoding='utf-8') as f:
            dados_brutos = json.load(f)
            print("Dados brutos recarregados do arquivo JSON.")
    
    # Cria o DataFrame
    df = pd.DataFrame(dados_brutos)
    print(f"DataFrame criado com sucesso. Dimensões iniciais: {df.shape[0]} linhas e {df.shape[1]} colunas.")
    
except FileNotFoundError:
    print("ERRO: O arquivo 'data/raw/projetos_df_raw.json' não foi encontrado. Execute a extração (Passo 3) novamente.")
    df = pd.DataFrame() # Cria um DataFrame vazio para evitar erros

DataFrame criado com sucesso. Dimensões iniciais: 10 linhas e 31 colunas.


### 3.2 Análise Inicial

Objetivo: Obter uma visão geral da estrutura do DataFrame, tipos de dados e estatísticas descritivas.



In [11]:
if not df.empty:
    # 1. Dimensões do DataFrame
    print("--- 1. Dimensões do DataFrame ---")
    print(f"O DataFrame possui {df.shape[0]} registros (projetos) e {df.shape[1]} atributos (colunas).")

    # 2. Tipos de Dados e Contagem de Não-Nulos
    print("\n--- 2. Tipos de Dados e Contagem de Não-Nulos (df.info()) ---")
    df.info()

    # 3. Amostra das Primeiras Linhas
    print("\n--- 3. Amostra das Primeiras Linhas (df.head()) ---")
    display(df.head())

    # 4. Lista de Colunas
    print("\n--- 4. Lista Completa de Colunas ---")
    print(df.columns.tolist())

    # 5. Estatísticas Descritivas
    print("\n--- 5. Estatísticas Descritivas (df.describe(include='all')) ---")
    # Usamos include='all' para obter estatísticas de colunas numéricas e categóricas
    display(df.describe(include='all').T)

--- 1. Dimensões do DataFrame ---
O DataFrame possui 10 registros (projetos) e 31 atributos (colunas).

--- 2. Tipos de Dados e Contagem de Não-Nulos (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   idUnico                             10 non-null     object
 1   nome                                10 non-null     object
 2   cep                                 6 non-null      object
 3   endereco                            6 non-null      object
 4   descricao                           10 non-null     object
 5   funcaoSocial                        10 non-null     object
 6   metaGlobal                          10 non-null     object
 7   dataInicialPrevista                 10 non-null     object
 8   dataFinalPrevista                   10 non-null     object
 9   dataInicialEfetiva     

,idUnico,nome,cep,endereco,descricao,funcaoSocial,metaGlobal,dataInicialPrevista,dataFinalPrevista,dataInicialEfetiva,...,observacoesPertinentes,isModeladaPorBim,dataSituacao,tomadores,executores,repassadores,eixos,tipos,subTipos,fontesDeRecurso
0,50379.53-54,DL - 304/2024 - Contratação de instituição par...,None,None,Contratação de instituição para execução de se...,Ampliação da capacidade de trafego visando a m...,Projetos Básicos e Executivos de Engenharia,2024-12-20,2027-12-05,None,...,None,False,2024-12-20,[],[{'nome': 'DEPARTAMENTO NACIONAL DE INFRAESTRU...,[],"[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 25, 'descricao': 'Rodovia', 'idEixo': 3}]","[{'id': 4, 'descricao': 'Acessos Terrestres', ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
1,42724.53-27,Escola Classe Crixá São Sebastião,None,None,"Construção de Escola em Tempo Integral, Escola...",A construção da nova escola beneficiará 977 es...,"Construção de Escola em Tempo Integral, Escola...",2024-09-02,2028-09-02,None,...,None,False,2025-09-05,[],[{'nome': 'SECRETARIA DE ESTADO DE EDUCACAO DO...,[{'nome': 'FUNDO NACIONAL DE DESENVOLVIMENTO D...,"[{'id': 4, 'descricao': 'Social'}]","[{'id': 46, 'descricao': 'Educação', 'idEixo':...","[{'id': 84, 'descricao': 'Educação', 'idTipo':...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
2,19970.53-78,Reajuste do Contrato 45/2021 - Contrução do Ce...,70.602-600,"SAIS Área Especial 3, Setor Policial Sul",Reajuste do Contrato 45/2021 - Construção do C...,Contribuir para a melhor formação dos bombeiro...,Construção de um novo centro de formação e de ...,2021-09-14,2024-08-28,None,...,None,False,2023-02-06,[],[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,[{'nome': 'CORPO DE BOMBEIROS MILITAR DO DISTR...,"[{'id': 1, 'descricao': 'Administrativo'}]","[{'id': 1, 'descricao': 'Segurança Pública', '...","[{'id': 59, 'descricao': 'Obras em Imóveis de ...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
3,24797.53-15,Implantação de Passarelas nas Estradas Parque ...,None,None,Implantação de passarelas de estrutura mista n...,"Pedestres, no geral, demanda das ocupações lin...",Implantação de passarelas de estrutura mista n...,2023-08-30,2028-08-30,None,...,None,False,2023-08-28,[],[{'nome': 'DEPARTAMENTO DE ESTRADAS DE RODAGEM...,"[{'nome': 'MINISTÉRIO DAS CIDADES', 'codigo': ...","[{'id': 3, 'descricao': 'Econômico'}]","[{'id': 24, 'descricao': 'Infraestrutura Urban...","[{'id': 57, 'descricao': 'Obra de Arte Especia...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."
4,24822.53-70,"obra de construção da Cabine de Medição, loca...",None,None,"obra de construção da Cabine de Medição, loca...",A demanda de carga elétrica do Campus Darcy Ri...,A demanda de carga elétrica do Campus Darcy Ri...,2023-09-14,2024-03-14,None,...,None,False,2023-08-29,[],"[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'nome': 'FUNDACAO UNIVERSIDADE DE BRASILIA',...","[{'id': 3, 'descricao': 'Econômico'}, {'id': 3...","[{'id': 31, 'descricao': 'Energia', 'idEixo': ...","[{'id': 95, 'descricao': 'Subestação', 'idTipo...","[{'origem': 'Federal', 'valorInvestimentoPrevi..."



--- 4. Lista Completa de Colunas ---
['idUnico', 'nome', 'cep', 'endereco', 'descricao', 'funcaoSocial', 'metaGlobal', 'dataInicialPrevista', 'dataFinalPrevista', 'dataInicialEfetiva', 'dataFinalEfetiva', 'dataCadastro', 'especie', 'natureza', 'naturezaOutras', 'situacao', 'descPlanoNacionalPoliticaVinculado', 'uf', 'qdtEmpregosGerados', 'descPopulacaoBeneficiada', 'populacaoBeneficiada', 'observacoesPertinentes', 'isModeladaPorBim', 'dataSituacao', 'tomadores', 'executores', 'repassadores', 'eixos', 'tipos', 'subTipos', 'fontesDeRecurso']

--- 5. Estatísticas Descritivas (df.describe(include='all')) ---


,count,unique,top,freq
idUnico,10,10,50379.53-54,1
nome,10,8,202111-22-Ronald 1,3
cep,6,4,70067-901,2
endereco,6,3,2021122-Ronald - Endereço Completo,3
descricao,10,8,2021122-Ronald - Descrição do Projeto,3
funcaoSocial,10,8,2021122-Ronald - Descrição Funç]ap Social,3
metaGlobal,10,8,20211122-Ronald - Descrição Meta Global,3
dataInicialPrevista,10,8,2021-12-10,3
dataFinalPrevista,10,8,2021-12-10,3
dataInicialEfetiva,0,0,NaN,NaN


### 3.3 - Análise de Qualidade

Objetivo: Quantificar a presença de valores nulos e identificar a existência de registros duplicados.



In [12]:
if not df.empty:
    # 1. Porcentagem de Valores Nulos por Coluna
    print("--- 1. Porcentagem de Valores Nulos por Coluna ---")
    nulos_perc = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    nulos_df = pd.DataFrame({'Nulos (%)': nulos_perc[nulos_perc > 0].round(2)})
    display(nulos_df)

    # 2. Identificação de Duplicatas
    print("\n--- 2. Identificação de Duplicatas ---")
    # A verificação de duplicatas no DataFrame completo falha devido às colunas aninhadas (listas/dicionários).
    # Focamos no identificador único do projeto.
    duplicatas_id = df['idUnico'].duplicated().sum()
    print(f"Total de linhas duplicadas (baseado no identificador 'idUnico'): {duplicatas_id}")
    
    # 3. Verificação de Valores Únicos em Colunas Chave
    print("\n--- 3. Verificação de Valores Únicos em Colunas Chave ---")
    for col in ['situacao', 'uf']:
        print(f"Coluna '{col}': {df[col].nunique()} valores únicos. Amostra: {df[col].unique()}")

--- 1. Porcentagem de Valores Nulos por Coluna ---


,Nulos (%)
dataFinalEfetiva,100.0
dataInicialEfetiva,100.0
observacoesPertinentes,100.0
qdtEmpregosGerados,90.0
populacaoBeneficiada,90.0
descPlanoNacionalPoliticaVinculado,80.0
descPopulacaoBeneficiada,80.0
naturezaOutras,60.0
endereco,40.0
cep,40.0



--- 2. Identificação de Duplicatas ---
Total de linhas duplicadas (baseado no identificador 'idUnico'): 0

--- 3. Verificação de Valores Únicos em Colunas Chave ---
Coluna 'situacao': 2 valores únicos. Amostra: ['Cadastrada' 'Cancelada']
Coluna 'uf': 1 valores únicos. Amostra: ['DF']


### 3.4 Síntese dos Problemas Encontrados


#### Problemas Estruturais e de Tipagem

*   **Colunas Aninhadas:** As colunas `executores`, `fontesDeRecurso`, `tipos`, etc., são do tipo `object` e contêm **listas de dicionários**. Isso é o principal obstáculo para a análise direta e exige o "achatamento" (flattening) dos dados.
*   **Tipagem de Datas:** As colunas de data (`dataInicialPrevista`, `dataFinalPrevista`, etc.) estão como `object` (string) e precisam ser convertidas para o tipo `datetime` para permitir cálculos temporais.

#### Problemas de Qualidade (Nulos)

*   **Alta Taxa de Nulos:** Colunas importantes como `dataInicialEfetiva` e `dataFinalEfetiva` (datas reais de início e fim) possuem uma alta taxa de valores nulos (100% no conjunto de 10 registros), indicando que a maioria dos projetos ainda está em fase de planejamento ou que a informação não está disponível na API.
*   **Localização:** Campos como `cep` e `endereco` também apresentam nulos, limitando a análise geoespacial.

#### Campos Chave e Plano de Ação

| Campo Chave | Problema Identificado | Plano de Ação (Transformação) |
| :--- | :--- | :--- |
| `fontesDeRecurso` | Lista de dicionários com o valor de investimento. | **Achatar:** Extrair e somar `valorInvestimentoPrevisto` para criar a coluna numérica `valor_total_previsto`. |
| `executores` | Lista de dicionários com o nome do órgão. | **Achatar:** Extrair o `nome` do órgão para criar a coluna categórica `orgao_executor`. |
| `data*` | Tipo `object` (string). | **Conversão:** Converter para o tipo `datetime`. |
| `idUnico` | Identificador único. | **Limpeza:** Usar para verificar e remover duplicatas (se houver). |
| `situacao` | Status do projeto. | **Limpeza:** Tratar nulos com "Não Informado" (se necessário). |


## 4. Tratamento de Dados
### 4.1 - Limpar Dados

Objetivo: Achatar as colunas aninhadas, remover duplicatas e normalizar os dados.



In [34]:
# --- Achatar Colunas Aninhadas e Extrair Valores ---

# Função para extrair o nome/descrição principal de uma lista de dicionários
def extract_main_info(list_of_dicts, key='nome'):
    if isinstance(list_of_dicts, list) and list_of_dicts:
        # Junta os nomes/descrições com um separador
        return '; '.join([str(d.get(key, '')) for d in list_of_dicts if isinstance(d, dict)])
    return None # Usar None para permitir o tratamento de nulos posterior

# Função para calcular o valor total previsto
def calculate_total_previsto(list_of_dicts):
    if isinstance(list_of_dicts, list):
        # Soma o valorInvestimentoPrevisto de todas as fontes de recurso
        return sum(d.get('valorInvestimentoPrevisto', 0) for d in list_of_dicts if isinstance(d, dict))
    return 0.0

# Aplicar as funções para criar novas colunas
df['orgao_executor'] = df['executores'].apply(lambda x: extract_main_info(x, key='nome'))
df['fonte_recurso'] = df['fontesDeRecurso'].apply(lambda x: extract_main_info(x, key='origem'))
df['tipo_projeto'] = df['tipos'].apply(lambda x: extract_main_info(x, key='descricao'))
df['valor_total_previsto'] = df['fontesDeRecurso'].apply(calculate_total_previsto)

# --- Seleção e Renomeação de Colunas (CORRIGIDO) ---

# Lista de colunas a serem selecionadas (removendo 'dataUltimaAtualizacao')
cols_to_select = [
    'idUnico', 'nome', 'situacao', 'dataInicialPrevista', 'dataFinalPrevista', 
    'dataInicialEfetiva', 'dataFinalEfetiva', 'dataCadastro',
    'orgao_executor', 'fonte_recurso', 'tipo_projeto', 'valor_total_previsto'
]

# Filtra a lista para incluir apenas as colunas que realmente existem no DataFrame
existing_cols = [col for col in cols_to_select if col in df.columns]

# Selecionar apenas as colunas que serão mantidas
df_clean = df[existing_cols].copy()

# Lista de novos nomes (deve corresponder à lista de colunas selecionadas)
new_names = [
    'id_unico', 'nome_projeto', 'status', 'data_inicio_prevista', 'data_fim_prevista',
    'data_inicio_real', 'data_fim_real', 'data_cadastro',
    'orgao_executor', 'fonte_recurso', 'tipo_projeto', 'valor_total_previsto'
]

# Ajusta a lista de novos nomes para corresponder às colunas que foram realmente selecionadas
df_clean.columns = new_names[:len(df_clean.columns)]

# --- Remover Duplicatas ---
df_clean.drop_duplicates(subset=['id_unico'], inplace=True)
print(f"Duplicatas removidas. Novo tamanho: {df_clean.shape[0]} linhas.")

# --- Normalizar Strings e Tratar Nulos Categóricos ---
# Normalizar strings (remover espaços e converter para maiúsculas/minúsculas)
for col in ['nome_projeto', 'status', 'orgao_executor', 'fonte_recurso', 'tipo_projeto']:
    # Tratar nulos categóricos com 'Não Informado'
    df_clean[col] = df_clean[col].fillna('Não Informado').astype(str).str.strip().str.upper()

# Tratar nulos em colunas que foram achatadas (usamos None na função, agora preenchemos)
df_clean['orgao_executor'] = df_clean['orgao_executor'].replace('NONE', 'NÃO INFORMADO')
df_clean['fonte_recurso'] = df_clean['fonte_recurso'].replace('NONE', 'NÃO INFORMADO')
df_clean['tipo_projeto'] = df_clean['tipo_projeto'].replace('NONE', 'NÃO INFORMADO')

Duplicatas removidas. Novo tamanho: 10 linhas.


### 4.2 - Conversão de Tipos

Objetivo: Garantir que as colunas estejam no formato correto para cálculos e visualizações.



In [35]:
# --- Converter Datas para Datetime ---
date_cols = [
    'data_inicio_prevista', 'data_fim_prevista', 'data_inicio_real', 
    'data_fim_real', 'data_cadastro'
]
for col in date_cols:
    # errors='coerce' transforma valores inválidos em NaT (Not a Time)
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce', utc=True).dt.tz_localize(None)

# --- Converter Valores Monetários para Float ---
df_clean['valor_total_previsto'] = pd.to_numeric(df_clean['valor_total_previsto'], errors='coerce').fillna(0.0)

# --- Categorizar Variáveis Apropriadas ---
for col in ['status', 'orgao_executor', 'fonte_recurso', 'tipo_projeto']:
    df_clean[col] = df_clean[col].astype('category')

### 4.3 - Feature Engineering

Objetivo: Criar novas colunas que enriquecem a análise.



In [36]:
# --- Ano/Mês de Início do Projeto (Previsto) ---
df_clean['ano_inicio_previsto'] = df_clean['data_inicio_prevista'].dt.year
df_clean['mes_inicio_previsto'] = df_clean['data_inicio_prevista'].dt.to_period('M')
df_clean['mes_inicio_previsto'] = df_clean['mes_inicio_previsto'].astype(str)
# --- Duração Estimada (em dias) ---
df_clean['duracao_prevista_dias'] = (df_clean['data_fim_prevista'] - df_clean['data_inicio_prevista']).dt.days

# --- Flags de Status (Binárias) ---
df_clean['is_concluido'] = df_clean['status'].str.contains('CONCLUIDA|CONCLUÍDA', na=False)
df_clean['is_cancelado'] = df_clean['status'].str.contains('CANCELADA', na=False)
df_clean['is_em_execucao'] = df_clean['status'].str.contains('EM EXECUÇÃO', na=False)

# --- Faixas de Valores (Binning) ---
bins = [0, 100000, 1000000, 10000000, df_clean['valor_total_previsto'].max() + 1]
labels = ['< 100K', '100K - 1M', '1M - 10M', '> 10M']
df_clean['faixa_valor'] = pd.cut(df_clean['valor_total_previsto'], bins=bins, labels=labels, right=False)

### 4.4 - Validar Dados Tratados

Objetivo: Confirmar que os problemas identificados foram resolvidos.



In [37]:
# 1. Verificar Ausência de Nulos Críticos
print("--- 1. Verificação de Nulos Críticos ---")
print("Nulos em 'orgao_executor':", df_clean['orgao_executor'].isnull().sum())
print("Nulos em 'valor_total_previsto':", df_clean['valor_total_previsto'].isnull().sum())

# 2. Confirmar Tipos Corretos
print("\n--- 2. Confirmação de Tipos (df.info()) ---")
df_clean.info()

# 3. Estatísticas Pós-Tratamento
print("\n--- 3. Estatísticas Descritivas Pós-Tratamento ---")
display(df_clean[['valor_total_previsto', 'duracao_prevista_dias']].describe())

# 4. Amostra Final
print("\n--- 4. Amostra Final do DataFrame Limpo ---")
display(df_clean.head())

--- 1. Verificação de Nulos Críticos ---
Nulos em 'orgao_executor': 0
Nulos em 'valor_total_previsto': 0

--- 2. Confirmação de Tipos (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_unico               10 non-null     object        
 1   nome_projeto           10 non-null     object        
 2   status                 10 non-null     category      
 3   data_inicio_prevista   10 non-null     datetime64[ns]
 4   data_fim_prevista      10 non-null     datetime64[ns]
 5   data_inicio_real       0 non-null      datetime64[ns]
 6   data_fim_real          0 non-null      datetime64[ns]
 7   data_cadastro          10 non-null     datetime64[ns]
 8   orgao_executor         10 non-null     category      
 9   fonte_recurso          10 non-null     category      
 10  tipo_projeto           10 non-null 

,valor_total_previsto,duracao_prevista_dias
count,1.000000e+01,10.000000
mean,4.336885e+07,599.300000
std,9.143939e+07,692.231344
min,7.000000e+05,0.000000
25%,1.708072e+06,38.000000
50%,1.155976e+07,197.000000
75%,3.000001e+07,1079.750000
max,3.000000e+08,1827.000000



--- 4. Amostra Final do DataFrame Limpo ---


,id_unico,nome_projeto,status,data_inicio_prevista,data_fim_prevista,data_inicio_real,data_fim_real,data_cadastro,orgao_executor,fonte_recurso,tipo_projeto,valor_total_previsto,ano_inicio_previsto,mes_inicio_previsto,duracao_prevista_dias,is_concluido,is_cancelado,is_em_execucao,faixa_valor
0,50379.53-54,DL - 304/2024 - CONTRATAÇÃO DE INSTITUIÇÃO PAR...,CADASTRADA,2024-12-20,2027-12-05,NaT,NaT,2024-12-20,DEPARTAMENTO NACIONAL DE INFRAESTRUTURA DE TRA...,FEDERAL,RODOVIA,44463443.00,2024,2024-12,1080,False,False,False,> 10M
1,42724.53-27,ESCOLA CLASSE CRIXÁ SÃO SEBASTIÃO,CANCELADA,2024-09-02,2028-09-02,NaT,NaT,2024-08-30,SECRETARIA DE ESTADO DE EDUCACAO DO DISTRITO F...,FEDERAL,EDUCAÇÃO,12319519.51,2024,2024-09,1461,False,True,False,> 10M
2,19970.53-78,REAJUSTE DO CONTRATO 45/2021 - CONTRUÇÃO DO CE...,CADASTRADA,2021-09-14,2024-08-28,NaT,NaT,2023-02-06,CORPO DE BOMBEIROS MILITAR DO DISTRITO FEDERAL,FEDERAL,SEGURANÇA PÚBLICA,1177429.91,2021,2021-09,1079,False,False,False,1M - 10M
3,24797.53-15,IMPLANTAÇÃO DE PASSARELAS NAS ESTRADAS PARQUE ...,CADASTRADA,2023-08-30,2028-08-30,NaT,NaT,2023-08-28,DEPARTAMENTO DE ESTRADAS DE RODAGEM DO DISTRIT...,FEDERAL,INFRAESTRUTURA URBANA E MOBILIDADE,10800000.00,2023,2023-08,1827,False,False,False,> 10M
4,24822.53-70,"OBRA DE CONSTRUÇÃO DA CABINE DE MEDIÇÃO, LOCA...",CADASTRADA,2023-09-14,2024-03-14,NaT,NaT,2023-08-29,FUNDACAO UNIVERSIDADE DE BRASILIA,FEDERAL,ENERGIA; ENERGIA,928139.70,2023,2023-09,182,False,False,False,100K - 1M


### 4.5 - Salvar Dados Tratados




In [38]:
import os
os.makedirs('data/processed', exist_ok=True)

# Salvar CSV processado
processed_file_path = 'data/processed/projetos_df_clean.csv'
df_clean.to_csv(processed_file_path, index=False, encoding='utf-8')
print(f"\nDados tratados salvos em: {processed_file_path}")


Dados tratados salvos em: data/processed/projetos_df_clean.csv


## 5. Armazenamento de Dados 

In [39]:
# Certifique-se de que estas bibliotecas estão instaladas:
# pip install sqlalchemy python-dotenv psycopg2-binary

from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import pandas as pd # Necessário para o DataFrame

# --- 1. Configuração e Conexão ---

# Carregar variáveis de ambiente do arquivo .env
load_dotenv()

# Construir a URL de Conexão para PostgreSQL
# O driver para PostgreSQL é 'postgresql+psycopg2'
db_url = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"

# Criar a Engine de Conexão
engine = create_engine(db_url)

# --- 2. Carga dos Dados ---

# Salvar DataFrame no PostgreSQL
# O método to_sql() faz a inserção
try:
    # if_exists='replace' garante que a tabela seja recriada a cada execução
    df_clean.to_sql('projetos', engine, if_exists='replace', index=False)
    
    print(f"Sucesso: {len(df_clean)} registros inseridos na tabela 'projetos' do PostgreSQL.")

    # --- 3. Verificação (Opcional) ---
    # Verifica se os dados foram realmente inseridos
    with engine.connect() as connection:
        count = connection.execute(text("SELECT COUNT(*) FROM projetos")).scalar_one()
        print(f"Verificação: {count} registros contados na tabela.")
        
except Exception as e:
    print(f"ERRO: Falha ao inserir dados no PostgreSQL. Verifique se o serviço está ativo, as permissões e as credenciais no .env. Erro: {e}")

Sucesso: 10 registros inseridos na tabela 'projetos' do PostgreSQL.
Verificação: 10 registros contados na tabela.


In [46]:
# Célula de Verificação no Jupyter Notebook

import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Carregar variáveis de ambiente
load_dotenv()

# Construir a URL de Conexão (usando as variáveis do .env)
db_url = f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

# Ler os dados da tabela 'projetos'
df_from_db = pd.read_sql('SELECT * FROM projetos', engine)

print(f"Dados lidos do PostgreSQL: {len(df_from_db)} registros.")
display(df_from_db.head())

Dados lidos do PostgreSQL: 10 registros.


,id_unico,nome_projeto,status,data_inicio_prevista,data_fim_prevista,data_inicio_real,data_fim_real,data_cadastro,orgao_executor,fonte_recurso,tipo_projeto,valor_total_previsto,ano_inicio_previsto,mes_inicio_previsto,duracao_prevista_dias,is_concluido,is_cancelado,is_em_execucao,faixa_valor
0,50379.53-54,DL - 304/2024 - CONTRATAÇÃO DE INSTITUIÇÃO PAR...,CADASTRADA,2024-12-20,2027-12-05,None,None,2024-12-20,DEPARTAMENTO NACIONAL DE INFRAESTRUTURA DE TRA...,FEDERAL,RODOVIA,44463443.00,2024,2024-12,1080,False,False,False,> 10M
1,42724.53-27,ESCOLA CLASSE CRIXÁ SÃO SEBASTIÃO,CANCELADA,2024-09-02,2028-09-02,None,None,2024-08-30,SECRETARIA DE ESTADO DE EDUCACAO DO DISTRITO F...,FEDERAL,EDUCAÇÃO,12319519.51,2024,2024-09,1461,False,True,False,> 10M
2,19970.53-78,REAJUSTE DO CONTRATO 45/2021 - CONTRUÇÃO DO CE...,CADASTRADA,2021-09-14,2024-08-28,None,None,2023-02-06,CORPO DE BOMBEIROS MILITAR DO DISTRITO FEDERAL,FEDERAL,SEGURANÇA PÚBLICA,1177429.91,2021,2021-09,1079,False,False,False,1M - 10M
3,24797.53-15,IMPLANTAÇÃO DE PASSARELAS NAS ESTRADAS PARQUE ...,CADASTRADA,2023-08-30,2028-08-30,None,None,2023-08-28,DEPARTAMENTO DE ESTRADAS DE RODAGEM DO DISTRIT...,FEDERAL,INFRAESTRUTURA URBANA E MOBILIDADE,10800000.00,2023,2023-08,1827,False,False,False,> 10M
4,24822.53-70,"OBRA DE CONSTRUÇÃO DA CABINE DE MEDIÇÃO, LOCA...",CADASTRADA,2023-09-14,2024-03-14,None,None,2023-08-29,FUNDACAO UNIVERSIDADE DE BRASILIA,FEDERAL,ENERGIA; ENERGIA,928139.70,2023,2023-09,182,False,False,False,100K - 1M


### Documentação da Estrutura do Banco de Dados (PostgreSQL)

#### Tabela Criada: `projetos`

A tabela `projetos` foi criada no esquema padrão (`public`) do PostgreSQL, utilizando o método `to_sql()` do Pandas.

#### Colunas e Tipos

As colunas da tabela `projetos` refletem a estrutura final do DataFrame `df_clean` após a transformação e *Feature Engineering*. O Pandas e o SQLAlchemy mapeiam os tipos de dados do Python para os tipos nativos do PostgreSQL.

| Coluna | Tipo de Dado (Pandas) | Tipo de Dado (PostgreSQL) | Descrição |
| :--- | :--- | :--- | :--- |
| `id_unico` | `object` | `TEXT` | Identificador único do projeto (Chave Primária Lógica). |
| `nome_projeto` | `category` | `VARCHAR` | Nome do projeto. |
| `status` | `category` | `VARCHAR` | Status atual do projeto (ex: CADASTRADA, CANCELADA). |
| `data_inicio_prevista` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de início prevista. |
| `data_fim_prevista` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de fim prevista. |
| `data_inicio_real` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de início efetiva (pode ser nula). |
| `data_fim_real` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de fim efetiva (pode ser nula). |
| `data_cadastro` | `datetime64[ns]` | `TIMESTAMP WITHOUT TIME ZONE` | Data de cadastro do projeto na API. |
| `orgao_executor` | `category` | `VARCHAR` | Órgão responsável pela execução (achatado). |
| `fonte_recurso` | `category` | `VARCHAR` | Fonte de recurso (ex: FEDERAL, ESTADUAL). |
| `tipo_projeto` | `category` | `VARCHAR` | Tipo de projeto (ex: RODOVIA, EDUCAÇÃO). |
| `valor_total_previsto` | `float64` | `DOUBLE PRECISION` | Valor total de investimento previsto (agregado). |
| `ano_inicio_previsto` | `float64` | `DOUBLE PRECISION` | Ano de início previsto (derivado). |
| `mes_inicio_previsto` | `object` | `TEXT` | Mês/Ano de início previsto (convertido para string para compatibilidade). |
| `duracao_prevista_dias` | `float64` | `DOUBLE PRECISION` | Duração estimada do projeto em dias (derivado). |
| `is_concluido` | `bool` | `BOOLEAN` | Flag: `True` se o status indica conclusão (derivado). |
| `is_cancelado` | `bool` | `BOOLEAN` | Flag: `True` se o status indica cancelamento (derivado). |
| `is_em_execucao` | `bool` | `BOOLEAN` | Flag: `True` se o status indica execução (derivado). |
| `faixa_valor` | `category` | `VARCHAR` | Faixa de valor do investimento (derivado). |

#### Por que Usar um Banco de Dados?

O uso de um banco de dados relacional como o PostgreSQL é fundamental para:

1.  **Persistência e Durabilidade:** Armazenar os dados de forma segura e estruturada, garantindo que o dataset limpo persista além da execução do script.
2.  **Consultas e Performance:** Permitir consultas SQL complexas e eficientes, que são mais rápidas e escaláveis do que operações em DataFrames muito grandes.
3.  **Conformidade ETL:** Atender ao requisito de Carga (Load) do pipeline ETL em um ambiente de produção, separando a lógica de processamento da lógica de armazenamento.


## 6. Análise Quantitativa

### 6.1 Perguntas de Negócio

A análise exploratória visa responder às seguintes perguntas de negócio, fornecendo uma visão sobre a distribuição e o foco dos investimentos no Distrito Federal:

a. **Qual a distribuição de valores de investimento?**  
   Identificar a faixa de valores mais comum e a presença de grandes projetos.

b. **Quais órgãos mais investem no DF?**  
   Análise do Top N órgãos executores por valor total previsto.

c. **Como se distribuem os projetos por tipo/categoria?**  
   Análise da proporção de projetos por `tipo_projeto`.

d. **Há evolução temporal nos investimentos?**  
   Análise da série temporal do valor de investimento previsto por ano.



### 6.2 - Análises Agregadas

Objetivo: Realizar os cálculos necessários para responder às perguntas de negócio.



In [65]:
# 1. Distribuição de Valores (Pergunta 1)
print("--- 1. Distribuição de Valores de Investimento (Quartis) ---")
print(df_clean['valor_total_previsto'].describe(percentiles=[.25, .5, .75, .95]))

# 2. Top N Órgãos Executores (Pergunta 2)
print("\n--- 2. Top 5 Órgãos Executores por Valor Total Previsto ---")
top_orgaos = (
    df_clean.groupby('orgao_executor', observed=True)['valor_total_previsto']
    .sum()
    .nlargest(5)
    .reset_index()
)
top_orgaos['percentual'] = (
    (top_orgaos['valor_total_previsto'] / df_clean['valor_total_previsto'].sum()) * 100
)
display(
    top_orgaos.style.format({
        'valor_total_previsto': 'R$ {:,.2f}',
        'percentual': '{:.2f}%'
    })
)
# 3. Distribuição por Tipo/Categoria (Pergunta 3)
print("\n--- 3. Distribuição de Projetos por Tipo/Categoria ---")
distribuicao_tipo = df_clean['tipo_projeto'].value_counts(normalize=True).mul(100).reset_index()
distribuicao_tipo.columns = ['tipo_projeto', 'percentual']
display(distribuicao_tipo.style.format({'percentual': '{:.2f}%'}))

# 4. Evolução Temporal (Pergunta 4)
print("\n--- 4. Evolução Temporal do Investimento Previsto por Ano ---")
investimento_anual = df_clean.groupby('ano_inicio_previsto')['valor_total_previsto'].sum().reset_index()
investimento_anual.columns = ['ano_inicio_previsto', 'valor_total_previsto']
display(investimento_anual.style.format({'valor_total_previsto': 'R$ {:,.2f}'}))



--- 1. Distribuição de Valores de Investimento (Quartis) ---
count    1.000000e+01
mean     4.336885e+07
std      9.143939e+07
min      7.000000e+05
25%      1.708072e+06
50%      1.155976e+07
75%      3.000001e+07
95%      1.850086e+08
max      3.000000e+08
Name: valor_total_previsto, dtype: float64

--- 2. Top 5 Órgãos Executores por Valor Total Previsto ---


,orgao_executor,valor_total_previsto,percentual
0,MINISTÉRIO DA INTEGRAÇÃO E DO DESENVOLVIMENTO REGIONAL,"R$ 360,000,015.00",83.01%
1,DEPARTAMENTO NACIONAL DE INFRAESTRUTURA DE TRANSPORTES,"R$ 44,463,443.00",10.25%
2,SECRETARIA DE ESTADO DE EDUCACAO DO DISTRITO FEDERAL,"R$ 12,319,519.51",2.84%
3,DEPARTAMENTO DE ESTRADAS DE RODAGEM DO DISTRITO FEDERAL,"R$ 10,800,000.00",2.49%
4,INSTITUTO FED. ED. CIENCIA E TEC. DE BRASILIA,"R$ 4,000,000.00",0.92%



--- 3. Distribuição de Projetos por Tipo/Categoria ---


,tipo_projeto,percentual
0,DESENVOLVIMENTO,30.00%
1,EDUCAÇÃO,30.00%
2,ENERGIA; ENERGIA,10.00%
3,INFRAESTRUTURA URBANA E MOBILIDADE,10.00%
4,RODOVIA,10.00%
5,SEGURANÇA PÚBLICA,10.00%



--- 4. Evolução Temporal do Investimento Previsto por Ano ---


,ano_inicio_previsto,valor_total_previsto
0,2021,"R$ 361,177,444.91"
1,2023,"R$ 15,728,139.70"
2,2024,"R$ 56,782,962.51"


### 6.3 - Estatísticas Descritivas

Objetivo: Aprofundar a análise estatística das variáveis numéricas.



In [41]:
# Medidas de Tendência Central e Dispersão para Valor e Duração
print("--- Estatísticas Descritivas Aprofundadas ---")
stats_descritivas = df_clean[['valor_total_previsto', 'duracao_prevista_dias']].describe().T
display(stats_descritivas.style.format('{:,.2f}'))

# Identificar Outliers (usando o IQR - Interquartile Range)
Q1 = df_clean['valor_total_previsto'].quantile(0.25)
Q3 = df_clean['valor_total_previsto'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR
limite_inferior = Q1 - 1.5 * IQR

outliers = df_clean[(df_clean['valor_total_previsto'] > limite_superior) | (df_clean['valor_total_previsto'] < limite_inferior)]
print(f"\nTotal de Outliers (Valor Previsto - IQR): {len(outliers)}")
display(outliers[['nome_projeto', 'valor_total_previsto']])

--- Estatísticas Descritivas Aprofundadas ---


,count,mean,std,min,25%,50%,75%,max
valor_total_previsto,10.00,"43,368,854.71","91,439,392.49","700,000.00","1,708,072.43","11,559,759.75","30,000,006.75","300,000,002.00"
duracao_prevista_dias,10.00,599.30,692.23,0.00,38.00,197.00,"1,079.75","1,827.00"



Total de Outliers (Valor Previsto - IQR): 1


,nome_projeto,valor_total_previsto
5,202111-22-RONALD 1,300000002.0


## 7. Visualizações

### 7.1 - Distribuição de Valores

Objetivo: Visualizar a distribuição do valor_total_previsto, identificando a concentração de projetos.



In [42]:
import plotly.express as px
import os

# Garantir que o diretório de visualizações exista
os.makedirs("visualizacoes", exist_ok=True)

# Excluir outliers (95% dos dados) para melhor visualização
q_high = df_clean['valor_total_previsto'].quantile(0.95)
df_filtered = df_clean[df_clean['valor_total_previsto'] <= q_high]

fig = px.histogram(df_filtered, x='valor_total_previsto',
                   title='1. Distribuição de Valores de Investimento (Excluindo 5% Maiores Outliers)',
                   labels={'valor_total_previsto': 'Valor Previsto (R$)'},
                   nbins=20) # Aumentar o número de bins para melhor detalhe

fig.update_layout(xaxis_title="Valor Total Previsto (R$)", yaxis_title="Contagem de Projetos")
fig.write_image("visualizacoes/01_distribuicao_valores.png")
fig.show()

### 7.2 - Análise por Órgão

Objetivo: Visualizar a contribuição dos órgãos executores para o investimento total.



In [71]:
# 1. Gráfico de Barras: Top 10 Órgãos por Valor
top_orgaos = (
    df_clean.groupby('orgao_executor', observed=True)['valor_total_previsto']
    .sum()
    .nlargest(10)
    .reset_index()
)

fig_bar = px.bar(
    top_orgaos,
    x='orgao_executor',
    y='valor_total_previsto',
    title='2. Top 10 Órgãos Executores por Valor de Investimento',
    labels={'orgao_executor': 'Órgão Executor', 'valor_total_previsto': 'Valor Total Previsto (R$)'},
    color='valor_total_previsto',
    color_continuous_scale=px.colors.sequential.Plasma
)

fig_bar.update_layout(xaxis={'categoryorder': 'total descending'})
fig_bar.write_image("visualizacoes/02_top_orgaos_barras.png")
fig_bar.show()

# 2. Gráfico de Pizza: Proporção do Total (Top 5 + Outros)
top_5_orgaos = df_clean['orgao_executor'].value_counts().nlargest(5).index.tolist()
df_pizza = df_clean.copy()
df_pizza['orgao_agregado'] = df_pizza['orgao_executor'].apply(lambda x: x if x in top_5_orgaos else 'OUTROS')

fig_pie = px.pie(
    df_pizza,
    names='orgao_agregado',
    values='valor_total_previsto',
    title='3. Proporção do Investimento por Órgão (Top 5 + Outros)',
    hole=.3
)

fig_pie.write_image("visualizacoes/03_proporcao_orgaos_pizza.png")
fig_pie.show()


### 7.3 - Análise Temporal

Objetivo: Visualizar a tendência de investimento ao longo do tempo.



In [44]:
# Série temporal de investimentos agregados por ano
df_time = df_clean.dropna(subset=['ano_inicio_previsto']).copy()
investimento_anual = df_time.groupby('ano_inicio_previsto')['valor_total_previsto'].sum().reset_index()

fig_line = px.line(investimento_anual, x='ano_inicio_previsto', y='valor_total_previsto',
                   title='4. Série Temporal do Investimento Previsto por Ano',
                   labels={'ano_inicio_previsto': 'Ano de Início Previsto', 'valor_total_previsto': 'Valor Previsto (R$)'},
                   markers=True)

fig_line.update_layout(xaxis_tickformat = 'd') # Formato de ano
fig_line.write_image("visualizacoes/04_evolucao_temporal.png")
fig_line.show()

### 7.4 - Análises Categóricas

Objetivo: Comparar a distribuição de valores entre diferentes categorias (ex: Status).



In [45]:
# Box Plot: Comparação de Valores por Status
fig_box = px.box(df_clean, x='status', y='valor_total_previsto',
                 title='5. Distribuição de Valores por Status do Projeto',
                 labels={'valor_total_previsto': 'Valor Previsto (R$)', 'status': 'Status do Projeto'})

fig_box.write_image("visualizacoes/05_box_status.png")
fig_box.show()

# Gráfico de Barras: Contagem de Projetos por Status
status_counts = df_clean['status'].value_counts().reset_index()
fig_status_bar = px.bar(status_counts, x='status', y='count',
                        title='6. Contagem de Projetos por Status',
                        labels={'count': 'Número de Projetos', 'status': 'Status do Projeto'})

fig_status_bar.write_image("visualizacoes/06_contagem_status.png")
fig_status_bar.show()

In [64]:

def load_and_process_data(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        dados_brutos = json.load(f)

    df = pd.DataFrame(dados_brutos)

    # Função para extrair a descrição principal de uma lista de dicionários
    def extract_main_info(list_of_dicts, key='descricao'):
        if isinstance(list_of_dicts, list) and list_of_dicts:
            unique_descriptions = sorted(list(set([str(d.get(key, '')) for d in list_of_dicts if isinstance(d, dict)])))
            return '; '.join(unique_descriptions)
        return None

    # Aplicar a função para achatar a coluna 'tipos'
    df['tipos_principais'] = df['tipos'].apply(lambda x: extract_main_info(x, key='descricao'))
    
    return df
# Carregar e processar os dados
df_processed = load_and_process_data('data/raw/projetos_df_raw.json')

# Contar a frequência de cada tipo de projeto
tipos_contagem = df_processed['tipos_principais'].value_counts().reset_index()
tipos_contagem.columns = ['Tipo de Projeto', 'Número de Projetos']

# Criar o gráfico de barras interativo com Plotly Express
fig = px.bar(tipos_contagem, 
             x='Tipo de Projeto', 
             y='Número de Projetos', 
             title='7. Distribuição de Projetos por Tipo',
             labels={'Tipo de Projeto': 'Tipo de Projeto', 'Número de Projetos': 'Número de Projetos'},
             color='Número de Projetos', # Colore as barras com base no número de projetos
             color_continuous_scale=px.colors.sequential.Viridis)



# Salvar o gráfico como PNG
fig.write_image("visualizacoes/07_distribuicao_tipos_projeto.png")

# Exibir o gráfico
fig.show()


## 8. Análise Qualitativa
### 8.1 Padrões Identificados na Análise

Com base nas análises quantitativas e visuais, os seguintes padrões foram observados:

1.  **Concentração de Investimentos (Padrão de Valor):**
    *   A distribuição de valores é altamente assimétrica, com a maioria dos projetos concentrada na faixa de valores mais baixos, mas o **valor total** é dominado por poucos projetos de grande porte (outliers).
    *   O **Top 5 Órgãos Executores** concentra a maior parte do valor total previsto, indicando que o investimento no DF é centralizado em poucas entidades.

2.  **Sazonalidade/Tendência Temporal:**
    *   A série temporal mostra que o investimento previsto não é constante, com picos em anos específicos (ex: 2024), sugerindo que o planejamento de grandes projetos é feito em ciclos ou em resposta a mandatos governamentais.

3.  **Status:**
    *   A maioria dos projetos  está em status de **CADASTRADA**, indicando que o volume de projetos em fase de planejamento é significativamente maior do que o volume em execução ou concluído.

### 8.2 Anomalias e Inconsistências


1.  **Inconsistências em Datas:**
    *   Projetos onde `data_inicio_prevista` é igual a `data_fim_prevista` (duração de 0 dias) são inconsistentes. Isso pode indicar um erro de preenchimento na API ou que o registro se refere a uma fase administrativa e não à execução física.

2.  **Status Questionáveis:**
    *   A presença de projetos com status **CANCELADA** merece investigação para entender as razões do cancelamento e o valor total de investimento perdido ou redirecionado.

### 8.3 Formulação de Hipóteses

1.  **Concentração de Órgãos:**
    *   **Hipótese:** Os órgãos que mais investem (ex: DNIT, Secretarias de Estado) possuem maior autonomia orçamentária e são responsáveis por projetos de infraestrutura de grande escala, que naturalmente demandam mais recursos.

2.  **Concentração Temporal:**
    *   **Hipótese:** Os picos de investimento previsto em anos específicos (ex: 2024) podem estar ligados ao **ciclo orçamentário plurianual** ou ao **início/fim de mandatos governamentais**, onde há um esforço concentrado para planejar e iniciar projetos.



## 9. Conclusão

Este projeto demonstrou a construção de um pipeline ETL (Extração, Transformação e Carga) completo, consumindo dados da API ObrasGov.br, tratando estruturas complexas (listas aninhadas) e persistindo o resultado em um banco de dados PostgreSQL. A análise exploratória revelou que o investimento previsto no Distrito Federal é altamente concentrado em poucos órgãos e tipos de projeto (Desenvolvimento e Educação), com um grande volume de projetos ainda na fase inicial de planejamento.

### Insights Principais

1.  **Concentração de Capital:** A maior parte do valor de investimento previsto está concentrada em poucos órgãos executores, destacando a necessidade de monitoramento focado nessas entidades.
2.  **Gargalo no Status:** A alta proporção de projetos no status **CADASTRADA** sugere um desafio na conversão do planejamento em execução, o que pode ser um foco para otimização de processos.
3.  **Prioridades de Investimento:** Os tipos de projeto **DESENVOLVIMENTO** e **EDUCAÇÃO** são as prioridades claras de investimento, refletindo o foco em infraestrutura e desenvolvimento social.

